---
title: "Wykład 6: Redukcja wymiarowości — PCA, t-SNE, UMAP"
format:
  html:
    embed-resources: true
    self-contained: true
---

## 6.1 Od pojedynczych zmiennych do przestrzeni wielowymiarowej

Przez dwa ostatnie wykłady pracowaliśmy z danymi, w których wymiary miały fizyczną interpretację: piksele ułożone w obraz, warstwy Z odpowiadające głębokości ogniskowania, klatki odpowiadające momentom czasu. Operacje takie jak `mean(axis=0)` miały intuicyjny sens — „uśredniamy po czasie."

Dziś wracamy do danych klinicznych z wykładów 1 i 3 — ale stawiamy pytanie, na które dotąd nie próbowaliśmy odpowiadać.

Na wykładzie 1 porównywaliśmy grupy pacjentów z kamicą i bez, zmienna po zmiennej: CRP, witamina D, BMI. Na wykładzie 3 testowaliśmy te różnice statystycznie — ponownie, jedna zmienna na raz. Ale każdy pacjent to nie jedna liczba. To **punkt w 38-wymiarowej przestrzeni**, gdzie każda oś to inna zmienna kliniczna. Być może grupy rozdzielają się dopiero wtedy, gdy spojrzymy na *kombinacje* zmiennych — a nie na każdą z osobna.

Pytanie brzmi: **czy pacjenci z kamicą żółciową i pacjenci zdrowi tworzą odrębne grupy, jeśli weźmiemy pod uwagę wszystkie zmienne naraz?**

Żeby na nie odpowiedzieć, potrzebujemy sposobu na zobaczenie 38 wymiarów na jednym wykresie. A tego, jak wiemy, zrobić się nie da — przynajmniej nie wprost.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('data/gallstone.csv')

# Usunięcie błędnych rekordów zidentyfikowanych w wykładzie 1
errors = [127, 178, 205, 239, 318]
df = df.drop(errors)

print(f"Rozmiar danych: {df.shape[0]} pacjentów, {df.shape[1]} zmiennych")

### 6.1.1 Pairplot — granica wizualizacji

Jednym ze sposobów na zobaczenie zależności między wieloma zmiennymi jest **pairplot** — macierz wykresów, w której każda para zmiennych dostaje swój panel. Spróbujmy z pięcioma wybranymi zmiennymi:

In [ ]:
selected = ['Body Mass Index (BMI)', 'C-Reactive Protein (CRP)',
            'Vitamin D', 'Total Cholesterol (TC)', 'Triglyceride',
            'Gallstone Status']

sns.pairplot(df[selected], hue='Gallstone Status',
             palette={0: 'steelblue', 1: 'indianred'},
             plot_kws={'alpha': 0.4, 's': 15})
plt.suptitle("Pairplot — 5 zmiennych", y=1.02)
plt.show()

Pięć zmiennych daje 10 paneli — to jeszcze do ogarnięcia, choć trzeba się przyjrzeć. Ale w naszym datasecie jest 38 zmiennych. Pairplot 38 zmiennych to $\frac{38 \times 37}{2} = 703$ panele. Nikt tego nie przeczyta. Potrzebujemy sposobu na „ściśnięcie" 38 wymiarów do 2 — zachowując jak najwięcej informacji o tym, jak pacjenci się od siebie różnią.


### 6.1.2 Biologiczna skala problemu

Nasze dane kliniczne mają 38 zmiennych. W biologii to niedużo. Dla porównania:

* **Genomika** — macierz ekspresji genów: 20 000 genów na próbkę. Pairplot potrzebowałby 200 milionów paneli.
* **Metabolomika** — profil metabolitów: setki do tysięcy związków chemicznych.
* **Single-cell RNA-seq** — profil transkryptomiczny pojedynczej komórki: ponad 10 000 genów.
* **Obrazy** — na wykładach 4–5 pracowaliśmy z obrazkami 196×171 pikseli. Każdy obraz to punkt w przestrzeni o 33 516 wymiarach.

We wszystkich tych przypadkach mamy ten sam problem: danych jest zbyt wiele wymiarów, żeby je zobaczyć. **Redukcja wymiarowości** to rodzina metod, które „ściskają" dane do 2–3 wymiarów, starając się zachować ich strukturę. Dziś poznamy trzy takie metody: PCA, t-SNE i UMAP.


## 6.2 PCA — Principal Component Analysis


### 6.2.1 Intuicja geometryczna

Zanim napiszemy jakikolwiek kod, pomyślmy geometrycznie.

Wyobraźmy sobie chmurę punktów w dwóch wymiarach — na przykład wzrost i waga 314 pacjentów. Punkty nie leżą losowo — tworzą wydłużoną elipsę, bo wzrost i waga są skorelowane (wyżsi ludzie ważą więcej).

PCA zadaje pytanie: **w jakim kierunku ta chmura jest najbardziej rozciągnięta?** Linia przebiegająca wzdłuż najdłuższej osi elipsy to **pierwsza składowa główna (PC1)**. Linia prostopadła do niej — to **PC2**. PCA obraca układ współrzędnych tak, żeby nowe osie pokrywały się z kierunkami największej zmienności w danych.

W dwóch wymiarach to brzmi trywialnie — ale zasada jest ta sama w 38 wymiarach. PCA szuka 38 nowych osi (składowych głównych), uporządkowanych od tej, wzdłuż której dane są najbardziej rozrzucone, do tej, wzdłuż której prawie się nie zmieniają. Jeśli pierwsze dwie składowe „łapią" większość zmienności, możemy zignorować resztę i narysować dane na płaszczyźnie PC1 × PC2.

**PCA nie usuwa danych — obraca perspektywę.** Zamiast patrzeć wzdłuż oryginalnych osi (BMI, CRP, cholesterol…), patrzymy wzdłuż osi, które najlepiej *różnicują* pacjentów.

Matematycznie PCA jest związane z rozkładem własnym macierzy kowariancji danych. Kierunek PC1 to wektor własny odpowiadający największej wartości własnej — czyli kierunek największej wariancji. PC2 to wektor własny z drugą co do wielkości wartością własną, i tak dalej.

### 6.2.2 Preprocessing: dlaczego standaryzacja jest konieczna

Zanim zastosujemy PCA, musimy rozwiązać praktyczny problem: nasze zmienne mają **zupełnie różne skale**.

In [ ]:
sample_vars = ['Height', 'Body Mass Index (BMI)',
               'C-Reactive Protein (CRP)', 'Visceral Fat Rating (VFR)',
               'Alkaline Phosphatase (ALP)', 'Creatinine']

df[sample_vars].describe().loc[['mean', 'std']].round(1)

Height ma średnią ~167 i std ~10. CRP ma średnią ~2 i std ~5. ALP ma średnią ~73 i std ~24. Skale różnią się dziesiątki razy. Dla PCA to katastrofa — metoda szuka kierunków największej *wariancji*, więc zmienna o dużej wariancji absolutnej (Height) zdominuje wynik, a zmienna o małej wariancji (CRP) — klinicznie ważna, jak wiemy z wykładu 3 — zniknie w szumie.

Rozwiązanie to **standaryzacja**: od każdej zmiennej odejmujemy jej średnią i dzielimy przez odchylenie standardowe. Po standaryzacji każda zmienna ma mean=0 i std=1, więc PCA traktuje je równo.

Robimy to na **zmiennych ciągłych**, pomijając dwie kategorie kolumn:

* **Gallstone Status** — to zmienna celu, nie cecha pacjenta. Nie jest pomiarem — jest diagnozą, którą chcemy *wyjaśnić*, nie *wkładać* do analizy. Za chwilę użyjemy jej do pokolorowania wykresu, ale PCA jej nie zobaczy.
* **Gender, Comorbidity, CAD, Hypothyroidism, Hyperlipidemia, DM, HFA** — to zmienne binarne (0/1). PCA zakłada ciągłość — zmienne binarne mają inną geometrię i wrzucanie ich razem z ciągłymi zaciemniłoby obraz.

To rozróżnienie jest ważne: Gallstone Status odpada, bo to etykieta. Zmienne binarne odpadają, bo PCA nie jest dla nich odpowiednią metodą. Dwa różne powody.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Zmienne binarne — pomijamy w PCA
binary_cols = ['Gallstone Status', 'Gender', 'Comorbidity',
               'Coronary Artery Disease (CAD)', 'Hypothyroidism',
               'Hyperlipidemia', 'Diabetes Mellitus (DM)',
               'Hepatic Fat Accumulation (HFA)']

feature_cols = [c for c in df.columns if c not in binary_cols]
print(f"Zmiennych ciągłych: {len(feature_cols)}")
print(f"Przykłady: {feature_cols[:5]}")

In [ ]:
X = df[feature_cols].values
y = df['Gallstone Status'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Przed standaryzacją — Height: mean={X[:, 0].mean():.1f}, std={X[:, 0].std():.1f}")
print(f"Po standaryzacji  — Height: mean={X_scaled[:, 0].mean():.4f}, std={X_scaled[:, 0].std():.4f}")

Po standaryzacji średnia jest praktycznie zerem, a odchylenie standardowe równe 1. Dotyczy to każdej kolumny — wszystkie zmienne startują z tej samej pozycji.

Metoda `fit_transform()` to skrót łączący dwa kroki, które w sklearn zawsze wyglądają tak samo:

* `fit(X)` — algorytm **uczy się** parametrów na danych (tu: oblicza średnią i odchylenie dla każdej kolumny),
* `transform(X)` — algorytm **stosuje** wyuczone parametry do danych (tu: odejmuje średnią, dzieli przez odchylenie).

Ten dwuetapowy wzorzec `fit` → `transform` jest uniwersalny w sklearn — spotkamy go zaraz przy PCA (`fit` uczy się kierunków największej wariancji, `transform` rzutuje dane na te kierunki), a na przyszłych przedmiotach — przy każdym modelu uczenia maszynowego. Rozdzielenie uczenia od stosowania ma praktyczny sens: parametry wyuczymy na jednym zbiorze danych, a zastosujemy do innego.

### 6.2.3 PCA na danych gallstone

Teraz możemy zastosować PCA. Zaczynamy od redukcji do 2 wymiarów — żeby zobaczyć dane na płaszczyźnie:

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Oryginalne wymiary: {X_scaled.shape}")
print(f"Po PCA:             {X_pca.shape}")

Z 31 zmiennych zostały 2 liczby na pacjenta. Narysujmy je:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for label, color, name in [(0, 'blue', 'Kontrola'),
                            (1, 'red', 'Kamica')]:
    mask = y == label
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=color, label=name, alpha=0.5, s=25)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} wariancji)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} wariancji)")
ax.set_title("PCA — dane kliniczne gallstone (zmienne ciągłe)")
ax.legend()
ax.grid(True);

**Co widzimy?**

Grupy się nakładają. Nie ma wyraźnej granicy między pacjentami z kamicą a kontrolą. Kamica żółciowa to choroba wieloczynnikowa, w której żaden pojedynczy profil metaboliczny nie oddziela chorych od zdrowych. Gdyby PCA dał dwa czyste klastry, powinniśmy być podejrzliwi — bo to oznaczałoby, że diagnoza jest trywialna, co kłóci się z wiedzą kliniczną.

Zwróćmy uwagę na coś istotnego: **PCA nie widziało etykiet.** Nie powiedzieliśmy algorytmowi, kto ma kamicę — daliśmy mu tylko 31 zmiennych ciągłych i poprosiliśmy o znalezienie kierunków największej wariancji. Kolory na wykresie to *nasza* decyzja analityczna — nakładamy je *po fakcie*, żeby sprawdzić, czy struktura odkryta przez PCA odpowiada czemuś, co znamy klinicznie.

Taki sposób pracy — metoda szuka struktury bez etykiet, a my sprawdzamy, czy ta struktura ma sens — nazywa się **uczeniem nienadzorowanym** (*unsupervised learning*). Wszystkie trzy metody na dzisiejszym wykładzie (PCA, t-SNE, UMAP) działają w ten sposób.

Na etykietach osi widzimy, ile wariancji wyjaśnia każda składowa. PC1 łapie 30%, a PC2 kolejne 22% — reszta informacji jest rozsiana po pozostałych komponentach. Tę kwestię zbadamy dokładniej za chwilę.


### 6.2.4 Scree plot — ile wymiarów zachować?

PCA z `n_components=2` to dopiero początek. Ile składowych *naprawdę* potrzeba, żeby oddać strukturę tych danych?

In [ ]:
pca_full = PCA().fit(X_scaled) # PCA bez ograniczenia liczby komponentów, żeby zobaczyć pełny rozkład wariancji

fig, ax = plt.subplots(figsize=(10, 5))
n = len(pca_full.explained_variance_ratio_)
x = range(1, n + 1)

ax.bar(x, pca_full.explained_variance_ratio_, alpha=0.6, label='Poszczególne PC')
ax.plot(x, np.cumsum(pca_full.explained_variance_ratio_), 'o-',
        color='red', markersize=4, label='Kumulatywnie')
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label='80% wariancji')
ax.set_xlabel("Numer komponentu")
ax.set_ylabel("Wyjaśniona wariancja")
ax.set_title("Scree plot — ile wymiarów zachować?")
ax.legend()
ax.set_xlim(0.5, n + 0.5)
plt.tight_layout()
plt.show()

**Interpretacja:** Scree plot pokazuje, ile informacji „łapie" każda kolejna składowa. Pierwsze składowe wyjaśniają najwięcej, potem szybko maleje — to typowy wzorzec. Próg 80% wariancji jest konwencją, nie dogmatem. Na potrzeby wizualizacji wystarczą 2–3 składowe. Do dalszej analizy (np. wejście do modelu predykcyjnego) zachowuje się tyle, ile potrzeba dla rozsądnej kompresji.

### 6.2.5 Loadings — co oznaczają PC1 i PC2?

PCA tworzy nowe osie jako **kombinacje liniowe** oryginalnych zmiennych. Współczynniki tych kombinacji nazywamy **loadings** — mówią, ile każda zmienna „wkłada" w dany komponent.

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_cols,
    columns=['PC1', 'PC2']
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

loadings['PC1'].sort_values().plot.barh(ax=axes[0], color='blue', alpha=0.7)
axes[0].set_title("Loadings — PC1")
axes[0].axvline(x=0, color='black', linewidth=0.5)

loadings['PC2'].sort_values().plot.barh(ax=axes[1], color='red', alpha=0.7)
axes[1].set_title("Loadings — PC2")
axes[1].axvline(x=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

**Próba interpretacji biologicznej:**

**PC1** można interpretować jako oś **ogólnego rozmiaru ciała**: po jednej stronie leżą razem wysokie loadings dla **masy mięśniowej, wody ustrojowej, masy ciała, masy kostnej** i części wskaźników tłuszczu trzewnego. To sugeruje, że ta składowa opisuje przede wszystkim przejście od osób **mniejszych i „lżejszych” somatycznie** do osób **większych, cięższych, o większej ilości tkanek i wody**. Ciekawy kontrast jest taki, że **HDL** leży po przeciwnej stronie osi, więc zmienia się odwrotnie niż ten ogólny wymiar „wielkości ciała”.

**PC2** jest jeszcze bardziej czytelna: to oś **„lean vs fat”**, czyli **beztłuszczowości/muskularności kontra otłuszczenia**. Po jednej stronie znajdują się **Lean Mass (%), Protein (%), wzrost, kreatynina, hemoglobina**, a po drugiej **TBFR, TFC, BMI, obesity %, VFA i VFR**. Innymi słowy, ta oś kontrastuje osoby o profilu bardziej **szczupłym i beztłuszczowym** z osobami o profilu bardziej **tłuszczowym, zwłaszcza z większym udziałem tłuszczu trzewnego**.

W PCA najważniejsze jest właśnie takie **kontrastowanie cech po dwóch stronach osi**: zmienne z tym samym znakiem zwykle współwystępują, a zmienne o znakach przeciwnych opisują **przeciwstawne wzorce biologiczne**. Trzeba też pamiętać, że **sam znak osi jest arbitralny** — można odwrócić całą składową bez zmiany interpretacji.

## 6.3 MNIST — od danych klinicznych do prawdziwej wysokiej wymiarowości

### 6.3.1 Obraz jako punkt w przestrzeni

Na wykładach 4 i 5 nauczyliśmy się, że obraz to macierz pikseli — tablica NumPy. Pracowaliśmy z obrazkami 196×171 pikseli, wyciągaliśmy z nich statystyki, stosowaliśmy maski i aggregacje.

Teraz odwrócimy perspektywę. Zamiast patrzeć *na* obraz, pomyślimy o obrazie jako o **jednym punkcie w przestrzeni wielowymiarowej**. Obraz 28×28 pikseli? Spłaszczamy go do wektora 784 liczb. To punkt w 784-wymiarowej przestrzeni. Zbiór takich punktów to dataset — i możemy go analizować tymi samymi narzędziami co dane kliniczne.

Użyjemy do tego **MNIST** — klasycznego datasetu ręcznie pisanych cyfr, jednego z najczęściej używanych zbiorów danych w uczeniu maszynowym.

In [ ]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_mnist = mnist.data.astype('float32')
y_mnist = mnist.target.astype(int)

print(f"Rozmiar: {X_mnist.shape}")
print(f"Cyfry:   {np.unique(y_mnist)}")

70 000 obrazków, każdy opisany przez 784 piksele. Zobaczmy kilka:

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(X_mnist[i].reshape(28, 28), cmap='gray')
    ax.set_title(f"{y_mnist[i]}", fontsize=10)
    ax.axis('off')
plt.suptitle("MNIST — każdy obraz to punkt w 784-wymiarowej przestrzeni", fontsize=13)
plt.tight_layout()
plt.show()

Każda cyfra to wektor 784 liczb. Dwie piątki leżą *blisko siebie* w tej przestrzeni — bo mają podobne piksele zaświecone w podobnych miejscach. Piątka i jedynka leżą *daleko* — bo wyglądają inaczej. Redukcja wymiarowości szuka sposobu, żeby tę bliskość zachować na płaskiej kartce.


### 6.3.2 PCA na MNIST — czy wystarczy?

Zanim sięgniemy po nowe metody, sprawdźmy, jak radzi sobie PCA:

In [ ]:
scaler_mnist = StandardScaler()
X_mnist_scaled = scaler_mnist.fit_transform(X_mnist)

pca_mnist = PCA(n_components=2)
X_mnist_pca = pca_mnist.fit_transform(X_mnist_scaled)

print(f"PC1 + PC2 wyjaśniają: {sum(pca_mnist.explained_variance_ratio_):.1%} wariancji")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(X_mnist_pca[:, 0], X_mnist_pca[:, 1],
                     c=y_mnist, cmap='tab10', s=0.5, alpha=0.3)
ax.set_xlabel(f"PC1 ({pca_mnist.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca_mnist.explained_variance_ratio_[1]:.1%})")
ax.set_title("PCA na MNIST — 784 wymiarów → 2")
plt.colorbar(scatter, label='Cyfra')
plt.tight_layout()
plt.show()

**Co widzimy?**

PCA daje pewne rozdzielenie — zera i jedynki wyodrębniają się na obrzeżach, bo ich kształty (okrągłe vs pionowe kreski) różnią się najbardziej wzdłuż liniowych kierunków. Ale większość cyfr tworzy jedną, nakładającą się chmurę. Czwórki, dziewiątki i siódemki mieszają się ze sobą, bo ich podobieństwo ma charakter *nieliniowy*.

PCA szuka prostych (liniowych) kierunków. Gdy struktura danych jest bardziej złożona — zakrzywiona, zwijająca się w wielu wymiarach — potrzebujemy metod, które potrafią „zginać" przestrzeń.

## 6.4 Metody nieliniowe: t-SNE i UMAP

### 6.4.1 Idea: zachowaj sąsiedztwo, nie odległości

PCA szuka osi maksymalnej wariancji. Metody nieliniowe zadają inne pytanie: **kto jest czyim sąsiadem?**

Intuicja jest prosta. Wyobraźmy sobie, że każdy punkt w $\mathbb{R}^{784}$ ma listę swoich najbliższych sąsiadów. Metoda nieliniowa próbuje ułożyć te 70 000 punktów w 2D tak, żeby każdy punkt miał *tych samych* sąsiadów co w oryginale. Nie próbuje zachować globalnych odległości (jak daleko jest od zera do dziewiątki) — skupia się na tym, żeby lokalna struktura (co leży obok czego) przetrwała ściskanie z $\mathbb{R}^{784}$ do $\mathbb{R}^2$.

To dlatego metody te potrafią „rozplątać" zakrzywione struktury, z którymi PCA sobie nie radzi.


### 6.4.2 t-SNE

Pierwszą z tych metod jest **t-SNE** (*t-distributed Stochastic Neighbor Embedding*), zaproponowana przez van der Maatena i Hintona w 2008 roku.

t-SNE na 70 000 próbek liczyłoby się kilka minut. Użyjemy losowego podzbioru — 7000 próbek to wystarczająco dużo, żeby zobaczyć strukturę, a obliczenia trwają kilkanaście sekund.

In [ ]:
from sklearn.manifold import TSNE

np.random.seed(42)
idx = np.random.choice(len(X_mnist), size=7000, replace=False)
X_sub = X_mnist_scaled[idx]
y_sub = y_mnist[idx]

print(f"Subsample: {X_sub.shape}")

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_sub)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1],
                     c=y_sub, cmap='tab10', s=3, alpha=0.6)
ax.set_title("t-SNE na MNIST (n=7000, perplexity=30)")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
plt.colorbar(scatter, label='Cyfra')
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

**Efekt jest dramatyczny.** Tam, gdzie PCA dawał amorficzną chmurę, t-SNE wyłania wyraźne wyspy — każda odpowiada innej cyfrze. Jedynki razem, piątki razem, zera razem. Metoda *sama* odkryła, że w tych danych istnieje 10 grup. Nie powiedzieliśmy jej, ile jest cyfr. Nie daliśmy etykiet. t-SNE po prostu zachowała sąsiedztwo — a grupy się wyłoniły.

**Pułapka 1: Odległości między klastrami nie mają sensu.**

Na wykresie t-SNE klaster jedynek może leżeć daleko od klastru dwójek. Ale to *nie oznacza*, że jedynki i dwójki są najbardziej różnymi cyframi. t-SNE zachowuje sąsiedztwo *wewnątrz* klastrów, ale odległości *między* klastrami są artefaktem optymalizacji, nie miarą podobieństwa.


**Pułapka 2: Parametr `perplexity` zmienia obraz.**

Perplexity to w przybliżeniu „ile sąsiadów bierze pod uwagę każdy punkt." Niska perplexity skupia się na bardzo lokalnej strukturze, wysoka — na bardziej globalnej. Zobaczmy to:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, perp in zip(axes, [5, 30, 100]):
    X_t = TSNE(n_components=2, perplexity=perp, random_state=42).fit_transform(X_sub)
    ax.scatter(X_t[:, 0], X_t[:, 1], c=y_sub, cmap='tab10', s=2, alpha=0.5)
    ax.set_title(f"perplexity = {perp}", fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])
plt.suptitle("t-SNE — wpływ perplexity", fontsize=14)
plt.tight_layout()
plt.show()

Przy perplexity=5 klastry się rozpadają na drobne podgrupy. Przy perplexity=100 zlewają się w większe struktury. Nie ma jednej „poprawnej" wartości — 30 to rozsądny punkt wyjścia, ale warto eksperymentować.


**Pułapka 3: Każdy run daje inny wynik.**

t-SNE jest stochastyczna — losowa inicjalizacja oznacza, że powtórzenie obliczeń (bez ustalenia `random_state`) da inny układ klastrów na wykresie. Klastry te same, ale ich rozmieszczenie — inne. Dlatego zawsze ustawiamy `random_state` i raportujemy go w publikacjach.


### 6.4.3 UMAP

**UMAP** (*Uniform Manifold Approximation and Projection*) to nowsza metoda (McInnes i in., 2018), która szybko stała się standardem — szczególnie w bioinformatyce, gdzie dominuje w wizualizacji single-cell RNA-seq.

Idea jest podobna do t-SNE: zachowaj sąsiedztwo. Ale UMAP ma dwie praktyczne przewagi: jest znacznie szybsza (na dużych zbiorach różnica bywa rzędu wielkości) i lepiej zachowuje **strukturę globalną** — odległości między klastrami mają w UMAP *trochę* więcej sensu niż w t-SNE.

In [ ]:
import umap

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=None)
X_umap = reducer.fit_transform(X_sub)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(X_umap[:, 0], X_umap[:, 1],
                     c=y_sub, cmap='tab10', s=3, alpha=0.6)
ax.set_title("UMAP na MNIST (n=7000)")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
plt.colorbar(scatter, label='Cyfra')
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

UMAP również wyłania wyraźne klastry cyfr — ale typowo daje je bardziej zwarte i lepiej odseparowane niż t-SNE. Warto zwrócić uwagę na relacje *między* klastrami: czy UMAP umieszcza jedynki blisko siódemek (podobne kształty — kreska) i daleko od zer (zupełnie inny kształt)?


**Dwa kluczowe parametry UMAP:**

* `n_neighbors` — ile sąsiadów rozpatrywać. Mała wartość (5) → drobna, lokalna struktura. Duża wartość (50) → gładka, globalna struktura.
* `min_dist` — jak ciasno pakować punkty. Mała wartość (0.0–0.1) → ciasne wyspy. Duża wartość (0.5–1.0) → rozproszone chmury.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
params = [(5, 0.1), (15, 0.1), (50, 0.5)]
for ax, (nn, md) in zip(axes, params):
    X_u = umap.UMAP(n_neighbors=nn, min_dist=md, random_state=42).fit_transform(X_sub)
    ax.scatter(X_u[:, 0], X_u[:, 1], c=y_sub, cmap='tab10', s=2, alpha=0.5)
    ax.set_title(f"n_neighbors={nn}, min_dist={md}", fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])
plt.suptitle("UMAP — wpływ parametrów", fontsize=14);

Mało sąsiadów i mała min_dist → drobne, rozbite klastry (UMAP „rozrywając" dane na szczegóły). Dużo sąsiadów i duża min_dist → gładsze, zlewające się struktury. Ustawienia domyślne (n_neighbors=15, min_dist=0.1) to rozsądny punkt startowy.


### 6.4.4 Porównanie trzech metod

Zestawmy wszystkie trzy podejścia obok siebie, na tych samych danych:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# PCA (na subsamplu, żeby porównanie było uczciwe)
axes[0].scatter(X_mnist_pca[idx, 0], X_mnist_pca[idx, 1],
                c=y_sub, cmap='tab10', s=2, alpha=0.4)
axes[0].set_title("PCA", fontsize=14)

# t-SNE
axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1],
                c=y_sub, cmap='tab10', s=2, alpha=0.4)
axes[1].set_title("t-SNE (perplexity=30)", fontsize=14)

# UMAP
axes[2].scatter(X_umap[:, 0], X_umap[:, 1],
                c=y_sub, cmap='tab10', s=2, alpha=0.4)
axes[2].set_title("UMAP (n_neighbors=15)", fontsize=14)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("784 wymiarów → 2: trzy podejścia", fontsize=15)
plt.tight_layout()
plt.show()

Różnica jest oczywista. PCA widzi pewną strukturę, ale grupy się nakładają. t-SNE i UMAP wyłuskują klastry, które odpowiadają cyfrom. Każda z metod ma swoje miejsce — pytanie brzmi, kiedy którą wybrać.

**PCA** stosujemy, gdy chcemy *zrozumieć* dane: co nowe osie znaczą (loadings), ile informacji zachowujemy (explained variance), jak zmienne się grupują. PCA jest deterministyczna (ten sam wynik za każdym razem), szybka i interpretowalna. Na danych liniowych (takich jak nasze zmienne kliniczne) radzi sobie dobrze.

**t-SNE** stosujemy, gdy chcemy *zobaczyć* klastry w danych nieliniowych. Świetna do wizualizacji, ale nie do dalszej analizy (osie nie mają sensu, odległości między klastrami — też nie).

**UMAP** to współczesny standard: szybsza od t-SNE, lepiej zachowuje strukturę globalną, skaluje się do dużych danych. W bioinformatyce stała się domyślnym narzędziem wizualizacji — praktycznie każdy artykuł o single-cell RNA-seq zawiera wykres UMAP.


## 6.5 Powrót do gallstone: czy UMAP ujawni coś, czego PCA nie pokazał?

Na początku wykładu pytaliśmy, czy pacjenci z kamicą i bez tworzą odrębne grupy. PCA pokazał, że nie — grupy się nakładają. Może UMAP, z jego zdolnością do wyłapywania nieliniowej struktury, pokaże coś innego?

In [ ]:
reducer_gs = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
X_gs_umap = reducer_gs.fit_transform(X_scaled)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, X_proj, title in [(axes[0], X_pca, "PCA"),
                           (axes[1], X_gs_umap, "UMAP")]:
    for label, color, name in [(0, 'b', 'Kontrola'),
                                (1, 'r', 'Kamica')]:
        mask = y == label
        ax.scatter(X_proj[mask, 0], X_proj[mask, 1],
                   c=color, label=name, alpha=0.5, s=20, edgecolors='none')
    ax.set_title(title, fontsize=14)
    ax.legend()
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("Gallstone — PCA vs UMAP", fontsize=15)
plt.tight_layout()
plt.show()

**Wynik jest pouczający.** UMAP nie daje dramatycznie lepszego rozdzielenia grup niż PCA. Mogą pojawić się lokalne zagęszczenia, być może jakaś substruktura — ale wyraźnej granicy kamica/kontrola nie będzie.

Na MNIST UMAP dał spektakularny efekt, bo dane miały wyraźną, nieliniową strukturę klastrową (10 odrębnych typów cyfr). Dane kliniczne gallstone tej struktury nie mają — korelacje między zmiennymi metabolicznymi są w przybliżeniu liniowe, a kamica to spektrum, nie dyskretna kategoria.

**Dobieramy metodę do danych, nie odwrotnie.** PCA na danych liniowych z interpretowalnymi zmiennymi. UMAP na danych wysokowymiarowych z nieliniową strukturą klastrową. Nie istnieje jedna metoda lepsza od wszystkich („no free lunch theorem") — wybieramy narzędzie, które najlepiej pasuje do charakteru naszych danych i pytania badawczego.


## 6.6 Podsumowanie


### Co umiemy po wykładzie 6

**Preprocessing:**

✅ `StandardScaler` — zawsze przed PCA (i t-SNE/UMAP) na danych o różnych skalach
✅ Separacja zmiennych ciągłych od binarnych — świadoma decyzja projektowa

**PCA:**

✅ Intuicja: obrót układu współrzędnych w kierunku największej wariancji
✅ `PCA(n_components=2).fit_transform(X_scaled)`
✅ Scree plot — ile komponentów zachować?
✅ Loadings — co oznaczają nowe osie? Interpretacja biologiczna
✅ Kiedy PCA wystarcza, a kiedy nie (liniowe vs nieliniowe dane)

**t-SNE:**

✅ Zachowuje lokalne sąsiedztwo, ale nie odległości globalne
✅ Parametr `perplexity` — wpływa na „ziarnistość" wizualizacji
✅ Stochastyczna — `random_state` dla powtarzalności
✅ Wolna na dużych danych → subsample
✅ Trzy pułapki: odległości, perplexity, losowość

**UMAP:**

✅ Szybsza alternatywa dla t-SNE, lepiej zachowuje strukturę globalną
✅ Parametry: `n_neighbors` (skala sąsiedztwa), `min_dist` (ciasność klastrów)
✅ Standard w bioinformatyce (single-cell, genomika)

**Kiedy co stosować:**

✅ PCA — eksploracja, interpretacja osi, raportowanie explained variance, dane liniowe
✅ t-SNE/UMAP — wizualizacja klastrów, dane nieliniowe, wysoka wymiarowość
✅ Nie istnieje jedna najlepsza metoda — dobieramy do danych


### Czego nie robiliśmy (i dlaczego)

Komponenty główne, które dziś wyciągaliśmy, mogą służyć jako cechy wejściowe do modeli predykcyjnych — zobaczycie to na Uczeniu maszynowym w semestrze 4. Nie robiliśmy tego dziś, bo redukcja wymiarowości jest wartościowa *sama w sobie* — jako narzędzie eksploracji i wizualizacji.

Nie wchodziliśmy też w matematykę PCA (dekompozycja SVD, formalne własności macierzy kowariancji) ani w teorię UMAP (topologia algebraiczna, fuzzy simplicial sets).


### Zapowiedź wykładu 7

Na dzisiejszym wykładzie redukowaliśmy wymiary danych numerycznych — klinicznych i pikselowych. Na następnym wykładzie przeniesiemy się do **danych sekwencyjnych**: aminokwasy, nukleotydy, motywy. Jak „zamienić" sekwencję białka na wektor liczb, który można wrzucić do PCA albo UMAP? I co nam powiedzą klastry w przestrzeni sekwencji o ewolucji i funkcji białek? Odpowie na to **Biopython** — i narzędzia, które dziś poznaliśmy.


### Źródła i dalsza lektura

**PCA:**
* Jake VanderPlas, „Python Data Science Handbook" — rozdział o PCA
* sklearn docs: https://scikit-learn.org/stable/modules/decomposition.html#pca

**t-SNE:**
* Wattenberg, Viégas, Johnson, „How to Use t-SNE Effectively" — interaktywna wizualizacja pułapek t-SNE: https://distill.pub/2016/misread-tsne/
* van der Maaten, Hinton (2008). „Visualizing Data using t-SNE." Journal of Machine Learning Research.

**UMAP:**
* Oficjalny tutorial: https://umap-learn.readthedocs.io/en/latest/basic_usage.html
* McInnes, Healy, Melville (2018). „UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction." arXiv:1802.03426.

**Dataset MNIST:**
* LeCun, Cortes, Burges — http://yann.lecun.com/exdb/mnist/
* Dostępny przez: `sklearn.datasets.fetch_openml('mnist_784')`


### Zadania do samodzielnej pracy

1. **Loadings detektyw:**
   Wykonaj `PCA(n_components=5)` na standaryzowanych danych gallstone. Dla każdego komponentu (PC1–PC5) zidentyfikuj 3 zmienne z najwyższymi loadings (wartość bezwzględna) i zaproponuj interpretację biologiczną — jaki proces lub cechę pacjenta opisuje dany komponent? Ile komponentów trzeba zachować, żeby wyjaśnić 80% wariancji?

2. **Perplexity explorer:**
   Uruchom t-SNE na MNIST (subsample 7000) z perplexity = 5, 10, 30, 50, 100. Dla każdej wartości policz, ile wizualnie odrębnych klastrów widzisz. Przy jakiej wartości klastry najlepiej odpowiadają rzeczywistym cyfrom? Czy istnieją cyfry, które są „trudne" (nakładają się z innymi) niezależnie od perplexity?

3. **UMAP na gallstone — kolorowanie według różnych zmiennych:**
   Zastosuj UMAP do danych gallstone. Narysuj ten sam wykres czterokrotnie, kolorując punkty kolejno według: (a) Gallstone Status, (b) Gender, (c) BMI (użyj `c=df.loc[idx, 'Body Mass Index (BMI)']` z `cmap='viridis'`), (d) Age. Która zmienna najlepiej „tłumaczy" układ punktów — czyli najwyraźniej segreguje kolory na wykresie?

4. **PCA vs UMAP na breast cancer:**
   Załaduj dataset `sklearn.datasets.load_breast_cancer()` (569 próbek × 30 cech, 2 klasy: złośliwy/łagodny). Standaryzuj dane, zastosuj PCA i UMAP, narysuj wykresy obok siebie z kolorami klas. Jak wypada porównanie z naszym gallstone — czy klasy się lepiej rozdzielają?

5. **Rekonstrukcja cyfry z N komponentów:**
   Zastosuj PCA z n_components = 2, 10, 50, 100, 200, 784 do MNIST. Dla jednej wybranej cyfry zrekonstruuj obraz z każdej liczby komponentów (użyj `pca.inverse_transform()`). Wyświetl 6 obrazów obok siebie. Ile komponentów potrzeba, żeby cyfra była rozpoznawalna? A ile — żeby wyglądała prawie jak oryginał?